# 🏀 March Madness Bracket Predictor
### Data Sources: KenPom · Bart Torvik · Evan Miya

This notebook:
1. Scrapes all three sites and exports individual CSVs
2. Merges data into a composite rating
3. Predicts win probabilities for every matchup
4. Simulates the full bracket and flags upsets

---

## ⚙️ Setup & Installs

In [ ]:
# Install dependencies if needed
# !pip install scrapy scrapy-playwright pandas numpy scipy
# !playwright install chromium

import os
import asyncio
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import expit
from IPython.display import display, HTML

# Create output directory for CSVs
Path('csvFiles').mkdir(exist_ok=True)
print('✅ Setup complete — csvFiles/ directory ready')

---
## 📡 Part 1 — Bart Torvik Scraper
Scrapes adjusted efficiency margin, tempo, strength of schedule, and more from **barttorvik.com**.
Exports to `csvFiles/torvik_data.csv`

In [ ]:
%%writefile torvik_spider.py
import scrapy
from scrapy_playwright.page import PageMethod


class TorvikSpider(scrapy.Spider):
    name = "torvik"

    custom_settings = {
        'TWISTED_REACTOR': 'twisted.internet.asyncioreactor.AsyncioSelectorReactor',
        'DOWNLOAD_HANDLERS': {
            'http':  'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
            'https': 'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
        },
        'FEEDS': {
            'csvFiles/torvik_data.csv': {'format': 'csv', 'overwrite': True}
        },
        'LOG_LEVEL': 'WARNING',
    }

    def start_requests(self):
        yield scrapy.Request(
            url='https://barttorvik.com/trank.php#',
            meta={
                'playwright': True,
                'playwright_include_page': True,
                'playwright_page_methods': [
                    PageMethod('wait_for_selector', 'tr.highlighted'),
                ],
            },
            callback=self.parse
        )

    async def parse(self, response):
        page = response.meta['playwright_page']
        for row in response.css('tr.highlighted'):
            team_name = row.css('td:nth-child(2) a::text').get()
            if not team_name:
                continue
            yield {
                'Team':      team_name.strip(),
                'Conf':      row.css('td:nth-child(3)::text').get('').strip(),
                'Record':    row.css('td:nth-child(4)::text').get('').strip(),
                'AdjOE':     row.css('td:nth-child(5)::text').get('').strip(),
                'AdjDE':     row.css('td:nth-child(6)::text').get('').strip(),
                'AdjEM':     row.css('td:nth-child(7)::text').get('').strip(),
                'AdjT':      row.css('td:nth-child(8)::text').get('').strip(),
                'Luck':      row.css('td:nth-child(9)::text').get('').strip(),
                'SOS_AdjEM': row.css('td:nth-child(10)::text').get('').strip(),
                'OppO':      row.css('td:nth-child(11)::text').get('').strip(),
                'OppD':      row.css('td:nth-child(12)::text').get('').strip(),
                'NCSOS':     row.css('td:nth-child(13)::text').get('').strip(),
            }
        await page.close()

In [ ]:
import subprocess, sys

print('🕷️ Running Torvik spider...')
result = subprocess.run(
    [sys.executable, '-m', 'scrapy', 'runspider', 'torvik_spider.py'],
    capture_output=True, text=True
)
if result.returncode == 0:
    torvik_df = pd.read_csv('csvFiles/torvik_data.csv')
    print(f'✅ Torvik data saved → csvFiles/torvik_data.csv  ({len(torvik_df)} teams)')
    display(torvik_df.head(10))
else:
    print('❌ Spider failed. Check output below:')
    print(result.stderr[-3000:])

---
## 📡 Part 2 — KenPom Scraper
Scrapes adjusted offensive/defensive efficiency, tempo, luck, and SOS from **kenpom.com**.

> ⚠️ **Requires a paid KenPom account.** Set `KENPOM_USER` and `KENPOM_PASS` env variables, or enter them in the cell below.

Exports to `csvFiles/kenpom_data.csv`

In [ ]:
# Set your KenPom credentials here (or use environment variables)
os.environ.setdefault('KENPOM_USER', 'YOUR_EMAIL_HERE')
os.environ.setdefault('KENPOM_PASS', 'YOUR_PASSWORD_HERE')

In [ ]:
%%writefile kenpom_spider.py
import scrapy, os
from scrapy_playwright.page import PageMethod


class KenpomSpider(scrapy.Spider):
    name = 'kenpom'

    custom_settings = {
        'TWISTED_REACTOR': 'twisted.internet.asyncioreactor.AsyncioSelectorReactor',
        'DOWNLOAD_HANDLERS': {
            'http':  'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
            'https': 'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
        },
        'FEEDS': {
            'csvFiles/kenpom_data.csv': {'format': 'csv', 'overwrite': True}
        },
        'LOG_LEVEL': 'WARNING',
    }

    def start_requests(self):
        yield scrapy.Request(
            url='https://kenpom.com/login.php',
            meta={'playwright': True, 'playwright_include_page': True},
            callback=self.do_login
        )

    async def do_login(self, response):
        page = response.meta['playwright_page']
        await page.fill('input[name="email"]',    os.environ.get('KENPOM_USER', ''))
        await page.fill('input[name="password"]', os.environ.get('KENPOM_PASS', ''))
        await page.click('input[type="submit"]')
        await page.wait_for_url('https://kenpom.com/', timeout=15000)
        yield scrapy.Request(
            url='https://kenpom.com/',
            meta={
                'playwright': True,
                'playwright_include_page': True,
                'playwright_page': page,
                'playwright_page_methods': [
                    PageMethod('wait_for_selector', 'table#ratings-table'),
                ],
            },
            callback=self.parse_ratings,
            dont_filter=True,
        )

    async def parse_ratings(self, response):
        page = response.meta['playwright_page']
        for row in response.css('table#ratings-table tbody tr'):
            team_name = row.css('td.team a::text').get()
            if not team_name:
                continue
            tds = row.css('td')
            def td(n): return tds[n].css('::text').get('').strip() if n < len(tds) else ''
            yield {
                'Team':      team_name.strip(),
                'Conf':      td(2),
                'Record':    td(3),
                'AdjEM':     td(4),
                'AdjO':      td(5),
                'AdjD':      td(6),
                'AdjT':      td(7),
                'Luck':      td(8),
                'SOS_AdjEM': td(9),
                'OppO':      td(10),
                'OppD':      td(11),
                'NCSOS':     td(12),
            }
        await page.close()

In [ ]:
if os.environ.get('KENPOM_USER') == 'YOUR_EMAIL_HERE':
    print('⚠️  KenPom credentials not set — skipping. Update KENPOM_USER/KENPOM_PASS above.')
else:
    print('🕷️ Running KenPom spider...')
    result = subprocess.run(
        [sys.executable, '-m', 'scrapy', 'runspider', 'kenpom_spider.py'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        kenpom_df = pd.read_csv('csvFiles/kenpom_data.csv')
        print(f'✅ KenPom data saved → csvFiles/kenpom_data.csv  ({len(kenpom_df)} teams)')
        display(kenpom_df.head(10))
    else:
        print('❌ Spider failed:')
        print(result.stderr[-3000:])

---
## 📡 Part 3 — Evan Miya Scraper
Scrapes Bayesian Performance Ratings (BPR), offensive/defensive BPR, and pace from **evanmiya.com**.
Uses a JS scroll trick to load all rows from the virtualized React table.

Exports to `csvFiles/evanmiya_data.csv`

In [ ]:
%%writefile evanmiya_spider.py
import scrapy
from scrapy import Selector
from scrapy_playwright.page import PageMethod


class EvanMiyaSpider(scrapy.Spider):
    name = 'evanmiya'

    custom_settings = {
        'TWISTED_REACTOR': 'twisted.internet.asyncioreactor.AsyncioSelectorReactor',
        'DOWNLOAD_HANDLERS': {
            'http':  'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
            'https': 'scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler',
        },
        'FEEDS': {
            'csvFiles/evanmiya_data.csv': {'format': 'csv', 'overwrite': True}
        },
        'LOG_LEVEL': 'WARNING',
        'PLAYWRIGHT_DEFAULT_NAVIGATION_TIMEOUT': 60000,
    }

    def start_requests(self):
        yield scrapy.Request(
            url='https://evanmiya.com/',
            meta={
                'playwright': True,
                'playwright_include_page': True,
                'playwright_page_methods': [
                    PageMethod('wait_for_selector', "div[role='rowgroup'] div[role='row']"),
                ],
            },
            callback=self.parse
        )

    async def parse(self, response):
        page = response.meta['playwright_page']

        # Scroll to load all rows in the virtualized table
        await page.evaluate("""
            async () => {
                const grid = document.querySelector("div[role='rowgroup']");
                if (!grid) return;
                let prev = -1;
                while (grid.scrollTop !== prev) {
                    prev = grid.scrollTop;
                    grid.scrollBy(0, 2000);
                    await new Promise(r => setTimeout(r, 400));
                }
            }
        """)

        content = await page.content()
        sel = Selector(text=content)

        for row in sel.css("div[role='rowgroup'] div[role='row']"):
            cells = row.css("div[role='gridcell']")
            if len(cells) < 8:
                continue
            def flat(n):
                return ' '.join(cells[n].css('::text').getall()).strip()
            team = flat(1)
            if not team:
                continue
            yield {
                'Team':   team,
                'Rank':   flat(0),
                'BPR':    flat(2),
                'OBPR':   flat(3),
                'DBPR':   flat(4),
                'Conf':   flat(5),
                'Record': flat(6),
                'Pace':   flat(7),
            }

        await page.close()

In [ ]:
print('🕷️ Running Evan Miya spider...')
result = subprocess.run(
    [sys.executable, '-m', 'scrapy', 'runspider', 'evanmiya_spider.py'],
    capture_output=True, text=True
)
if result.returncode == 0:
    evanmiya_df = pd.read_csv('csvFiles/evanmiya_data.csv')
    print(f'✅ Evan Miya data saved → csvFiles/evanmiya_data.csv  ({len(evanmiya_df)} teams)')
    display(evanmiya_df.head(10))
else:
    print('❌ Spider failed:')
    print(result.stderr[-3000:])

---
## 🔗 Part 4 — Merge All Three Sources
Loads the three CSVs, normalises team names, and builds a weighted composite rating.

| Source | Weight | Metric Used |
|--------|--------|-------------|
| KenPom | 40% | AdjEM |
| Torvik | 35% | AdjEM |
| Evan Miya | 25% | BPR |

In [ ]:
def load_csv_if_exists(path):
    if Path(path).exists():
        df = pd.read_csv(path)
        df['Team'] = df['Team'].str.strip().str.title()
        return df
    return None

torvik_df   = load_csv_if_exists('csvFiles/torvik_data.csv')
kenpom_df   = load_csv_if_exists('csvFiles/kenpom_data.csv')
evanmiya_df = load_csv_if_exists('csvFiles/evanmiya_data.csv')

sources_loaded = [name for name, df in
    [('Torvik', torvik_df), ('KenPom', kenpom_df), ('EvanMiya', evanmiya_df)]
    if df is not None]
print(f'✅ Loaded sources: {", ".join(sources_loaded)}')

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std()

# Start with Torvik as the base (always available)
merged = torvik_df[['Team', 'Conf', 'Record', 'AdjOE', 'AdjDE', 'AdjEM', 'AdjT', 'SOS_AdjEM']].copy()
merged = merged.rename(columns={'AdjEM': 'TV_AdjEM', 'AdjOE': 'TV_AdjOE', 'AdjDE': 'TV_AdjDE'})

# Merge KenPom if available
if kenpom_df is not None:
    kp = kenpom_df[['Team', 'AdjEM', 'AdjO', 'AdjD']].rename(
        columns={'AdjEM': 'KP_AdjEM', 'AdjO': 'KP_AdjO', 'AdjD': 'KP_AdjD'})
    merged = merged.merge(kp, on='Team', how='left')
else:
    merged['KP_AdjEM'] = np.nan

# Merge EvanMiya if available
if evanmiya_df is not None:
    em = evanmiya_df[['Team', 'BPR', 'OBPR', 'DBPR']].rename(
        columns={'BPR': 'EM_BPR', 'OBPR': 'EM_OBPR', 'DBPR': 'EM_DBPR'})
    merged = merged.merge(em, on='Team', how='left')
else:
    merged['EM_BPR'] = np.nan

# Convert all numeric columns
num_cols = [c for c in merged.columns if c not in ('Team', 'Conf', 'Record')]
for col in num_cols:
    merged[col] = pd.to_numeric(merged[col], errors='coerce')

# Weighted composite rating (z-score normalised)
tv_z = zscore(merged['TV_AdjEM'].fillna(merged['TV_AdjEM'].median()))
kp_z = zscore(merged['KP_AdjEM'].fillna(merged['TV_AdjEM'].median()))  # fallback to torvik if missing
em_z = zscore(merged['EM_BPR'].fillna(merged['EM_BPR'].median()))

if kenpom_df is not None:
    merged['CompositeRating'] = 0.40 * kp_z + 0.35 * tv_z + 0.25 * em_z
    print('Composite weights: KenPom 40% | Torvik 35% | EvanMiya 25%')
else:
    merged['CompositeRating'] = 0.60 * tv_z + 0.40 * em_z
    print('Composite weights: Torvik 60% | EvanMiya 40%  (KenPom not available)')

merged = merged.sort_values('CompositeRating', ascending=False).reset_index(drop=True)
merged.index += 1  # 1-based rank
merged.index.name = 'CompositeRank'

# Save merged CSV for others to use
merged.to_csv('csvFiles/merged_ratings.csv')
print(f'✅ Merged data saved → csvFiles/merged_ratings.csv  ({len(merged)} teams)')

display(merged[['Team', 'Conf', 'Record', 'TV_AdjEM', 'KP_AdjEM', 'EM_BPR', 'CompositeRating']].head(25))

---
## 🎯 Part 5 — Win Probability Model

Uses a logistic (sigmoid) function on composite rating differences.
Calibrated so a **+10 composite advantage ≈ 75% win probability**, consistent with historical tournament results.

In [ ]:
K = 0.35  # calibration constant

# Build team → composite rating lookup
ratings = dict(zip(merged['Team'], merged['CompositeRating']))

def win_prob(team_a: str, team_b: str) -> float:
    """Return probability that team_a beats team_b."""
    r_a = ratings.get(team_a, 0.0)
    r_b = ratings.get(team_b, 0.0)
    return float(expit(K * (r_a - r_b)))

def is_upset(seed_a: int, seed_b: int, prob_a: float) -> bool:
    """Flag as upset if lower-seeded team (worse) wins with >35% prob."""
    return (seed_b - seed_a) >= 5 and prob_a > 0.35

# Quick sanity check
print('Win probability model ready.')
print('Sanity check — top team vs median team:')
top_team   = merged.iloc[0]['Team']
mid_team   = merged.iloc[len(merged)//2]['Team']
prob = win_prob(top_team, mid_team)
print(f'  {top_team} vs {mid_team}: {prob:.1%} / {1-prob:.1%}')

---
## 🗓️ Part 6 — Define the 2025 Bracket

Edit the `BRACKET` dict below to match the actual tournament field. Teams must match the names in the scraped data.

In [ ]:
# Format: { region: [(seed, 'Team Name'), ...] }
# Teams are listed in bracket order — pairs play each other: (1 vs 16), (8 vs 9), etc.

BRACKET = {
    'East': [
        (1, 'Duke'),              (16, 'American'),
        (8, 'Mississippi State'), (9,  'Boise State'),
        (5, 'Oregon'),            (12, 'Liberty'),
        (4, 'Arizona'),           (13, 'Akron'),
        (6, 'Byu'),               (11, 'Vcu'),
        (3, 'Wisconsin'),         (14, 'Montana'),
        (7, "Saint Mary'S"),      (10, 'Vanderbilt'),
        (2, 'Alabama'),           (15, 'Robert Morris'),
    ],
    'West': [
        (1, 'Florida'),           (16, 'Norfolk State'),
        (8, 'Connecticut'),       (9,  'Oklahoma'),
        (5, 'Memphis'),           (12, 'Colorado State'),
        (4, 'Maryland'),          (13, 'Grand Canyon'),
        (6, 'Missouri'),          (11, 'Drake'),
        (3, 'Texas Tech'),        (14, 'Mcneese State'),
        (7, 'Kansas'),            (10, 'Arkansas'),
        (2, "St. John'S"),        (15, 'Omaha'),
    ],
    'South': [
        (1, 'Auburn'),            (16, 'Alabama State'),
        (8, 'Louisville'),        (9,  'Creighton'),
        (5, 'Michigan'),          (12, 'Uc San Diego'),
        (4, 'Texas A&M'),         (13, 'Yale'),
        (6, 'Ole Miss'),          (11, 'San Diego State'),
        (3, 'Iowa State'),        (14, 'Lipscomb'),
        (7, 'Marquette'),         (10, 'New Mexico'),
        (2, 'Michigan State'),    (15, 'Bryant'),
    ],
    'Midwest': [
        (1, 'Houston'),           (16, 'Siue'),
        (8, 'Gonzaga'),           (9,  'Georgia'),
        (5, 'Clemson'),           (12, 'Mcneese'),
        (4, 'Purdue'),            (13, 'High Point'),
        (6, 'Illinois'),          (11, 'Texas'),
        (3, 'Kentucky'),          (14, 'Troy'),
        (7, 'Ucla'),              (10, 'Utah State'),
        (2, 'Tennessee'),         (15, 'Wofford'),
    ],
}

print('✅ Bracket defined — 4 regions × 16 teams = 64 teams total')

# Check for any teams missing from scraped data
all_bracket_teams = [team for region in BRACKET.values() for _, team in region]
missing = [t for t in all_bracket_teams if t not in ratings]
if missing:
    print(f'\n⚠️  Teams not found in ratings data (will use 0.0 rating):')
    for t in missing:
        print(f'   - {t}')
else:
    print('✅ All bracket teams found in ratings data!')

---
## 🏆 Part 7 — Simulate the Full Bracket

In [ ]:
ROUND_NAMES = {
    1: 'Round of 64',
    2: 'Round of 32',
    3: 'Sweet 16',
    4: 'Elite 8',
    5: 'Final Four',
    6: 'Championship'
}

def simulate_region(region_name, teams_seeds):
    all_matchups = []
    current = list(teams_seeds)
    round_num = 1
    while len(current) > 1:
        next_round = []
        for i in range(0, len(current), 2):
            seed_a, team_a = current[i]
            seed_b, team_b = current[i + 1]
            prob_a = win_prob(team_a, team_b)
            winner = current[i] if prob_a >= 0.5 else current[i + 1]
            all_matchups.append({
                'Region':     region_name,
                'Round':      round_num,
                'RoundName':  ROUND_NAMES[round_num],
                'Team_A':     team_a,  'Seed_A': seed_a,
                'Team_B':     team_b,  'Seed_B': seed_b,
                'WinProb_A':  round(prob_a, 4),
                'WinProb_B':  round(1 - prob_a, 4),
                'Winner':     winner[1],
                'WinnerSeed': winner[0],
                'Upset':      is_upset(seed_a, seed_b, prob_a) or
                              is_upset(seed_b, seed_a, 1 - prob_a),
            })
            next_round.append(winner)
        current = next_round
        round_num += 1
    return all_matchups, current[0]  # matchups + region winner


all_results = []
region_winners = {}

print('Simulating regions...')
for region, teams in BRACKET.items():
    matchups, winner = simulate_region(region, teams)
    all_results.extend(matchups)
    region_winners[region] = winner
    print(f'  {region:8s} → {winner[1]:25s} (#{winner[0]} seed)')

# Final Four — East vs West, South vs Midwest
print('\nSimulating Final Four...')
ff_pairs = [('East', 'West'), ('South', 'Midwest')]
ff_winners = []
for r1, r2 in ff_pairs:
    s1, t1 = region_winners[r1]
    s2, t2 = region_winners[r2]
    prob = win_prob(t1, t2)
    winner = (s1, t1) if prob >= 0.5 else (s2, t2)
    ff_winners.append(winner)
    all_results.append({
        'Region': f'Final Four ({r1} vs {r2})', 'Round': 5,
        'RoundName': 'Final Four',
        'Team_A': t1, 'Seed_A': s1, 'Team_B': t2, 'Seed_B': s2,
        'WinProb_A': round(prob, 4), 'WinProb_B': round(1-prob, 4),
        'Winner': winner[1], 'WinnerSeed': winner[0],
        'Upset': is_upset(s1, s2, prob) or is_upset(s2, s1, 1-prob),
    })
    print(f'  {t1} vs {t2}  →  {winner[1]} ({prob:.0%} / {1-prob:.0%})')

# Championship
(s1, t1), (s2, t2) = ff_winners
prob = win_prob(t1, t2)
champion = (s1, t1) if prob >= 0.5 else (s2, t2)
all_results.append({
    'Region': 'Championship', 'Round': 6,
    'RoundName': 'Championship',
    'Team_A': t1, 'Seed_A': s1, 'Team_B': t2, 'Seed_B': s2,
    'WinProb_A': round(prob, 4), 'WinProb_B': round(1-prob, 4),
    'Winner': champion[1], 'WinnerSeed': champion[0],
    'Upset': is_upset(s1, s2, prob) or is_upset(s2, s1, 1-prob),
})

print(f'\n🏆 PREDICTED CHAMPION: {champion[1]} (#{champion[0]} seed)')

---
## 💾 Part 8 — Save Results & Export CSVs

In [ ]:
results_df = pd.DataFrame(all_results)
upsets_df  = results_df[results_df['Upset'] == True].copy()

# Save all CSVs
results_df.to_csv('csvFiles/bracket_predictions.csv', index=False)
upsets_df.to_csv('csvFiles/upset_picks.csv', index=False)

print('CSV files saved:')
print('  csvFiles/torvik_data.csv          — raw Torvik ratings')
print('  csvFiles/kenpom_data.csv          — raw KenPom ratings (if scraped)')
print('  csvFiles/evanmiya_data.csv        — raw Evan Miya ratings')
print('  csvFiles/merged_ratings.csv       — all sources merged + composite rating')
print('  csvFiles/bracket_predictions.csv  — full bracket round-by-round')
print('  csvFiles/upset_picks.csv          — flagged upset predictions')

---
## 📊 Part 9 — Results Explorer

In [ ]:
# Full bracket by round
for round_num in sorted(results_df['Round'].unique()):
    rname = ROUND_NAMES[round_num]
    subset = results_df[results_df['Round'] == round_num]
    print(f'\n{'='*60}')
    print(f'  {rname}')
    print(f'{'='*60}')
    for _, row in subset.iterrows():
        upset_flag = ' 🚨 UPSET' if row['Upset'] else ''
        print(f"  #{row['Seed_A']:2d} {row['Team_A']:22s} {row['WinProb_A']:.0%}  |  "
              f"{row['WinProb_B']:.0%} #{row['Seed_B']:2d} {row['Team_B']:22s}  "
              f"→ {row['Winner']}{upset_flag}")

In [ ]:
# Upset picks summary
print(f'⚠️  UPSET PICKS ({len(upsets_df)} total)\n')
display(
    upsets_df[['RoundName', 'Region', 'Winner', 'WinnerSeed', 'Team_A', 'Seed_A',
               'Team_B', 'Seed_B', 'WinProb_A', 'WinProb_B']]
    .sort_values(['Round', 'WinnerSeed'])
    .reset_index(drop=True)
)

In [ ]:
# Top 25 composite ratings
print('Top 25 Teams by Composite Rating\n')
display(
    merged[['Team', 'Conf', 'Record', 'CompositeRating', 'TV_AdjEM', 'KP_AdjEM', 'EM_BPR']]
    .head(25)
    .style.background_gradient(subset=['CompositeRating'], cmap='RdYlGn')
)